# TB Portals — Spatial Cavity Attention Overlays (Figure F1)
Loads the saved `SpatialCavityHead` checkpoints from the agentic A2 run, picks 2 cavity-positive + 1 cavity-negative test image per country, and saves attention heatmap overlays on the raw CXRs.

Attach: `tb-portals-cxr-pngs`. Internet **ON**. GPU T4. Runtime ≈ 15 min.

## 0 — Clone repo + install deps

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = '/kaggle/working/dl-project-codebase'
BRANCH = 'cleaned-repo'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
for p in (REPO_DIR, REPO_DIR + '/scripts'):
    if p not in sys.path: sys.path.insert(0, p)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'matplotlib'], check=False)
print('ready')

## 1 — Run the A2 best+spat-cav config briefly to (a) cache features, (b) re-train head, (c) save attention weights per test image
We can't run the saved-head from local zip because it was saved without per-image attention exposure. Easier path: re-train R4b spatial cavity head briefly (~2 min total), then use it to compute attention on a curated case list.

In [ ]:
import os
WORK = '/kaggle/working'
REPO_DIR = '/kaggle/working/dl-project-codebase'
DATASET = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = f'{DATASET}/kaggle_export'
PAPER_MANIFEST = f'{WORK}/tbportals_manifest_paper.csv'
FEATURES_CLS  = f'{WORK}/features_rad-dino_cls.npz'
FEATURES_GRID = f'{WORK}/features_rad-dino_grid7.npz'

# build manifest
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + '/scripts' not in sys.path: sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample
raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv', dtype={'image_id': str, 'patient_id': str, 'country': str})
raw['image_path'] = raw['image_path'].apply(lambda p: p if str(p).startswith('/') else f'{KAGGLE_EXPORT}/{p}')
paper_df = subsample(raw, seed=42)
paper_df['image_id'] = paper_df['image_path'].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print('manifest size:', len(paper_df))

In [ ]:
# cache features (CLS + grid)
from cache_features import main as cache_main
for path, grid in [(FEATURES_CLS, 0), (FEATURES_GRID, 7)]:
    if os.path.isfile(path):
        print('cached ->', path); continue
    args = ['--manifest', PAPER_MANIFEST, '--out', path,
            '--backbone', 'rad-dino', '--batch-size', '32']
    if grid > 0:
        args += ['--patch-grid', str(grid)]
    cache_main(args)

In [ ]:
# Train the R4b spatial cavity head per country (seed 0 only, fast)
from src.training.train_agentic import main as agentic_main
outdir = f'{WORK}/cavattn_run'
if not os.path.isdir(outdir):
    agentic_main(['--features', FEATURES_CLS, '--features-grid', FEATURES_GRID,
                  '--manifest', PAPER_MANIFEST,
                  '--mode', 'a2', '--out-dir', outdir,
                  '--rungs', '4', '--seeds', '0',
                  '--held-outs', 'Romania', 'Moldova', 'Kazakhstan',
                  '--save-heads'])
print('done')

## 2 — Generate the 9 attention-overlay PNGs

In [ ]:
import torch, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib as mpl
from src.components.feature_heads import SpatialCavityHead
from cache_features import load_features
from src.data.tbportals_dataset import _load_image
from src.data.tbportals import make_country_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
grid_feats, dim = load_features(FEATURES_GRID)
manifest = pd.read_csv(PAPER_MANIFEST, dtype={'image_id': str})
id_to_path = dict(zip(manifest['image_id'], manifest['image_path']))
id_to_cav  = dict(zip(manifest['image_id'], manifest['cavity']))
id_to_alp  = dict(zip(manifest['image_id'], manifest['alp_0_100']))
id_to_country = dict(zip(manifest['image_id'], manifest['country']))

OUT_DIR = Path(f'{WORK}/cavity_attn_pngs')
OUT_DIR.mkdir(exist_ok=True)

def overlay(image_path: str, attn49: np.ndarray, title: str, out_path: Path):
    img = np.asarray(Image.open(image_path).convert('L'), dtype=np.float32) / 255.0
    H, W = img.shape
    attn_grid = attn49.reshape(7, 7)
    fig, ax = plt.subplots(figsize=(3.2, 3.4))
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.imshow(attn_grid, extent=(0, W, H, 0), alpha=0.5, cmap='inferno',
              vmin=attn_grid.min(), vmax=attn_grid.max(), interpolation='bilinear')
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(pad=0.2)
    plt.savefig(out_path, dpi=140, bbox_inches='tight')
    plt.close()

rows_meta = []
for country in ['Romania', 'Moldova', 'Kazakhstan']:
    head_path = Path(outdir) / 'heads' / f'a2_rung4b_spatial_cavity_{country}_s0.pt'
    if not head_path.exists():
        print('missing head ->', head_path); continue
    blob = torch.load(head_path, map_location=device)
    head = SpatialCavityHead(in_dim=dim).to(device).eval()
    head.load_state_dict(blob['cav_head'])

    # pick 2 positives + 1 negative from this country's TEST split
    _, _, test_df = make_country_split(manifest, held_out_country=country, val_fraction=0.2, seed=0)
    test_ids = [i for i in test_df['image_id'].astype(str).values if i in grid_feats]
    test_pos = [i for i in test_ids if int(id_to_cav.get(i, 0)) == 1]
    test_neg = [i for i in test_ids if int(id_to_cav.get(i, 0)) == 0]
    # rank positives by predicted cavity prob descending; negatives ascending
    def score(ids):
        Xg = np.stack([grid_feats[i] for i in ids])
        with torch.no_grad():
            xt = torch.from_numpy(Xg).float().to(device)
            logit = head(xt)        # [N, 2]
            prob = torch.softmax(logit, dim=1)[:, 1].cpu().numpy()
        return prob
    pp = score(test_pos)
    pn = score(test_neg)
    pick_pos = [test_pos[i] for i in np.argsort(-pp)[:2]]
    pick_neg = [test_neg[i] for i in np.argsort(pn)[:1]]

    for tag, ids in [('pos', pick_pos), ('neg', pick_neg)]:
        for img_id in ids:
            Xg = torch.from_numpy(grid_feats[img_id][None]).float().to(device)
            with torch.no_grad():
                logit, attn = head(Xg, return_attn=True)
                attn49 = attn[0].cpu().numpy()
                prob = float(torch.softmax(logit, dim=1)[0, 1].cpu())
            label = int(id_to_cav.get(img_id, 0))
            alp = float(id_to_alp.get(img_id, 0))
            title = f'{country} {tag}: GT cav={label} ALP={alp:.0f} | pred={prob:.2f}'
            png_path = OUT_DIR / f'{country}_{tag}_{img_id[-8:]}.png'
            overlay(id_to_path[img_id], attn49, title, png_path)
            rows_meta.append({'country': country, 'tag': tag, 'image_id': img_id,
                              'pred_prob': prob, 'gt_cavity': label,
                              'gt_alp': alp, 'png': str(png_path)})
    print(f'{country}: rendered 3 PNGs')

meta_df = pd.DataFrame(rows_meta)
meta_df.to_csv(OUT_DIR / 'meta.csv', index=False)
print('wrote', len(rows_meta), 'PNGs')

## 3 — Bundle and download

In [ ]:
import shutil
zip_path = shutil.make_archive(f'{WORK}/cavity_attn_pngs', 'zip', str(OUT_DIR))
print('zip ->', zip_path)